# PASTORS + TabPFN (Google Colab)

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

Upload to a Drive folder (e.g. `MyDrive/pastors/`) keeping this structure:
```
pastors/
  helpers/                         (all .py)
  data/geospatial/oman_processed_closed.geojson
  data/geospatial/validation_processed_closed.geojson
  data/satellite/fused_spot_s2_oman.tif (+ .json + _metadata.json)
  output/baresoil_mask_oman.tif        (only for raster prediction)
```
Set `BASE` below to that folder.

In [1]:
!pip install -q tabpfn-client rasterio geopandas composition_stats

In [2]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/pastors'     # <-- your Drive folder
os.chdir(BASE); sys.path.insert(0, BASE)

# --- TabPFN cloud client: load token from local file (git-ignored) ---
import json as _json
from tabpfn_client import set_access_token
with open('./utils/tabpfn_token.json') as _f:
    _tok = _json.load(_f)['token']
set_access_token(_tok)
print("TabPFN client authenticated")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TabPFN client authenticated


In [3]:
import numpy as np, pandas as pd, geopandas as gpd
from helpers.prepare_validation import prepare_validation
from helpers.background_removal import background_removal
from helpers.load_satellite_data import load_satellite_data
from helpers.symmetric_balances import group_balance
from helpers.dendro_groups import compositional_groups
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from tabpfn_client import TabPFNRegressor
import pandas as pd

# ── Config ──
dataset   = "kenya" #"kenya", "oman"
satellite = f"fused_spot_s2_{dataset}"
buffer    = 500
SHARPEN_SWIR          = False
RESTRICT_TRAINING     = True
USE_GLOBAL_BACKGROUND = False
AUTO_GROUPS       = True
if dataset == "oman":
    ACTIVE_SITES        = ["S4","S7","S10"]
    GROUP_EXCLUDE_SITES = ["S11","S12"]
    GROUP_TYPE_PREFIX   = "CORRAL"
else:  # kenya
    ACTIVE_SITES        = ["E2","E4","E7","E9"]   # pick your enclosure sites
    GROUP_EXCLUDE_SITES = []
    GROUP_TYPE_PREFIX   = "COR"   # matches both CORAL and CORRAL

# fallback manual groups if AUTO_GROUPS=False
MARKERS_MANUAL    = ["P","K","Ca"]
NATBG_MANUAL      = ["Si","Al","Fe","Ti","Mg"]

In [4]:
# ── Load, background-remove, detect groups, build balance ──
gdf  = gpd.read_file(f"./data/geospatial/{dataset}_processed_closed.geojson")

VAL_PATH = "./data/geospatial/validation_processed_closed.geojson"
HAS_VAL  = (dataset == "oman") and os.path.exists(VAL_PATH)

if HAS_VAL:
    gval = gpd.read_file(VAL_PATH)
    gval["Serial"] = gval["Serial"].astype(str)
    gdf, gval_bg, shared = prepare_validation(gdf, gval, restrict_training=RESTRICT_TRAINING)
else:
    gval_bg = None
    print("No validation set — skipping external validation")

if AUTO_GROUPS:
    grp = compositional_groups(f"./data/geospatial/{dataset}_processed_closed.geojson",
            include_sites=ACTIVE_SITES, exclude_sites=GROUP_EXCLUDE_SITES,
            type_prefix=GROUP_TYPE_PREFIX, exclude_parts=("Res",))
    markers, natural_bgk = grp["markers"], grp["natural_bgk"]
else:
    markers, natural_bgk = MARKERS_MANUAL, NATBG_MANUAL
print("markers:", markers, "| natural_bgk:", natural_bgk)

gbk = background_removal(gdf, buffer_distance=buffer, use_global_background=USE_GLOBAL_BACKGROUND)
df  = load_satellite_data(satellite, gbk, sharpen_swir=SHARPEN_SWIR)
df["balance"] = group_balance(df, markers, natural_bgk)

if HAS_VAL:
    gval_bk = background_removal(gval_bg, buffer_distance=buffer, use_global_background=USE_GLOBAL_BACKGROUND)
    gval_bk = gval_bk.loc[gval_bk.index.intersection(gval_bg["Serial"])]
    dfv = load_satellite_data(satellite, gval_bk, sharpen_swir=SHARPEN_SWIR)
    dfv["balance"] = group_balance(dfv, markers, natural_bgk)

No validation set — skipping external validation
markers: ['Mg', 'P', 'K', 'Ca', 'Mn', 'Zn', 'Rb', 'Sr', 'Zr'] | natural_bgk: ['Al', 'Si', 'Ti', 'Fe', 'Y', 'Ba']
✅ Background removal complete for 20 sites.
Loading raster: ./data/satellite/fused_spot_s2_kenya.tif
  aggregated 23 multi-sample pixels
  added NDVI, SAVI
  added BSI, NDMI, Clay_Index
  added NDWI, Calcite_Index
  output: 1603 rows x 37 cols (16 elements + 14 bands + 7 indices)
✅ Satellite data integration complete
✅ Completed balance comparing between marker-elements and natural-background.


In [5]:
# ── Feature matrix ──
sat_kw = ("Band_","Drag_","S2_","SPOT_","Maxar_")
idx_kw = ("NDVI","SAVI","BSI","NDMI","NDWI","MSAVI","Clay","Calcite","Iron","EVI","NBR")
feature_cols = [c for c in df.columns if c.startswith(sat_kw) or any(c.startswith(k) for k in idx_kw)]
X = df[feature_cols].values; y = df["balance"].values
print(f"features: {len(feature_cols)} | train samples: {len(y)} | val: {len(dfv) if HAS_VAL else 0}")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

features: 21 | train samples: 1603 | val: 0


In [6]:
reg = TabPFNRegressor(n_estimators=8)
reg.fit(X_tr, y_tr)

def metrics(yt, yp):
    return dict(R2=round(r2_score(yt,yp),3),
                RMSE=round(np.sqrt(mean_squared_error(yt,yp)),3),
                MAE=round(mean_absolute_error(yt,yp),3),
                Pearson=round(pearsonr(yt,yp)[0],3),
                Spearman=round(spearmanr(yt,yp)[0],3))
print("Internal 20% test:", metrics(y_te, reg.predict(X_te)))

00:09 Fitting... Done!
00:08 Predicting... Done!
Internal 20% test: {'R2': 0.311, 'RMSE': np.float64(0.229), 'MAE': 0.127, 'Pearson': np.float64(0.559), 'Spearman': np.float64(0.528)}


In [7]:
# ── External validation (predict directly on validation features) ──
reg_full = TabPFNRegressor(n_estimators=8).fit(X, y)

if HAS_VAL:
    Xv = dfv[feature_cols].values; yv = dfv["balance"].values
    pred_v = reg_full.predict(Xv)

    # uncertainty: 90% predictive interval
    lo, med, hi = reg_full.predict(Xv, output_type="quantiles", quantiles=[0.05, 0.5, 0.95])
    unc_v = hi - lo

    m = np.isfinite(pred_v) & np.isfinite(yv)
    print("External validation:", metrics(yv[m], pred_v[m]))

    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,6)); plt.scatter(yv[m], pred_v[m], edgecolor="k")
    lim=[min(yv[m].min(),pred_v[m].min()), max(yv[m].max(),pred_v[m].max())]
    plt.plot(lim,lim,"r--"); plt.xlabel("True balance"); plt.ylabel("Predicted")
    plt.title(f"{dataset} external validation (TabPFN)"); plt.show()
else:
    print("No validation set — reg_full fitted on all training data for raster prediction")

00:05 Fitting... Done!
No validation set — reg_full fitted on all training data for raster prediction


In [8]:
import rasterio
from helpers.load_satellite_data import _add_spectral_indices, _read_band_names

def predict_raster_tabpfn(reg, satellite, feature_cols, mu, sd, y_lo, y_hi,
                          base="./data/satellite", mask_path=None, sharpen_swir=False,
                          batch=200000, unc_pct=95, out_path=None):
    out_path = out_path or f"./output/{dataset}_tabpfn_prediction.tif"
    with rasterio.open(f"{base}/{satellite}.tif") as src:
        prof = src.profile.copy(); img = src.read().astype("float32")
        bn = _read_band_names(satellite, base, src.count); H, W = src.height, src.width
    if sharpen_swir:
        from helpers.feature_ops import sharpen_swir_bands; img = sharpen_swir_bands(img, bn)
    B = img.shape[0]
    dfx = _add_spectral_indices(pd.DataFrame(img.reshape(B, -1).T, columns=bn), satellite)
    Xr = dfx[feature_cols].values
    valid = np.isfinite(Xr).all(1)
    if mask_path:
        with rasterio.open(mask_path) as mk:
            valid &= (mk.read(1).reshape(-1) == 1)
    idx = np.where(valid)[0]
    print(f"predicting {len(idx):,} bare-soil pixels of {Xr.shape[0]:,}")

    pred = np.full(Xr.shape[0], np.nan, "float32")
    unc  = np.full(Xr.shape[0], np.nan, "float32")
    for i in range(0, len(idx), batch):
        j = idx[i:i+batch]
        Xs = (Xr[j] - mu) / sd                      # same scaling as training
        pred[j] = reg.predict(Xs)
        lo, med, hi = reg.predict(Xs, output_type="quantiles", quantiles=[0.05, 0.5, 0.95])
        unc[j] = hi - lo
        print(f"  {min(i+batch, len(idx)):,}/{len(idx):,}")

    pred = np.where(np.isfinite(pred), np.clip(pred, y_lo, y_hi), np.nan).astype("float32")
    thr = np.nanpercentile(unc, unc_pct)            # mask the most-uncertain pixels
    pred_masked = pred.copy(); pred_masked[unc > thr] = np.nan
    print(f"masking pixels with 90% interval width > {thr:.3f} (top {100-unc_pct:.0f}%)")

    prof.update(count=1, dtype="float32", nodata=np.nan, compress="lzw")
    os.makedirs("./output", exist_ok=True)
    def _w(arr, path):
        with rasterio.open(path, "w", **prof) as d: d.write(arr.reshape(H, W), 1)
    _w(pred, out_path)
    _w(unc, out_path.replace(".tif", "_uncertainty.tif"))
    _w(pred_masked, out_path.replace(".tif", "_masked.tif"))
    print("saved ->", out_path, "(+ _uncertainty, + _masked)")
    return out_path

predict_raster_tabpfn(reg_full, satellite, feature_cols, mu_f, sd_f, Y_LO, Y_HI,
                      mask_path=f"./output/baresoil_mask_{dataset}.tif",
                      sharpen_swir=SHARPEN_SWIR)

  added NDVI, SAVI
  added BSI, NDMI, Clay_Index
  added NDWI, Calcite_Index
predicting 2,001,898 bare-soil pixels of 3,539,808
00:24 Predicting... Done!
00:00 Predicting... /

00:13 Predicting... Done!
  200,000/2,001,898
00:25 Predicting... Done!
00:00 Predicting... /

00:14 Predicting... Done!
  400,000/2,001,898
00:21 Predicting... Done!
00:00 Predicting... /

00:13 Predicting... Done!
  600,000/2,001,898
00:24 Predicting... Done!
00:00 Predicting... /

00:13 Predicting... Done!
  800,000/2,001,898
00:23 Predicting... Done!
00:00 Predicting... /

...

00:13 Predicting... Done!
  1,600,000/2,001,898
00:26 Predicting... Done!
00:00 Predicting... -

00:14 Predicting... Done!
  1,800,000/2,001,898
00:38 Predicting... Done!
00:00 Predicting... /

00:13 Predicting... Done!
  2,000,000/2,001,898
00:04 Predicting... Done!
00:00 Predicting... \

00:01 Predicting... Done!
  2,001,898/2,001,898
saved -> ./output/kenya_tabpfn_prediction.tif


'./output/kenya_tabpfn_prediction.tif'